<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/09_1_Function_Calling_and_Custom_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[실습 09-1] Function Calling과 Custom Tool 연동  

### 실습목표

- 여러 개의 매개변수(Arguments)를 가진 파이썬 함수를 Tool로 정의하고 에이전트에 등록할 수 있다.  

- LLM이 자연어 문장에서 도구 실행에 필요한 상세 정보(도시명, 단위 등)를 추출하는 과정을 관찰한다.  

- 실제 API 호출 결과(Observation)를 바탕으로 LLM이 최종 답변을 재구성하는 흐름을 이해한다.  

1. 실습 준비 및 도구 정의  

- 에이전트가 사용할 수 있는 LangChain, Google API 라이브러리를 설치하고 Import하며, Google API Key를 설정합니다.  

In [ ]:
# 기존 설치를 무시하고 최신 버전으로 강제 재설치합니다.
!pip install -q -U --force-reinstall langchain langchain-community langchain-huggingface langchain-core langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
# Google API Key 설정
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API 설정 완료")

Gemini API 설정 완료


In [ ]:
# Import LangChain 및 Google API
import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_core.messages import ToolMessage

2. Custom tool 정의  
2.1 단순 Custom tool (계산기)  

In [ ]:
# 1 Custom Tool 정의 (계산기)
@tool
def complex_calculator(expression: str) -> str:
    """복잡한 수학 계산이나 수식을 처리할 때 사용합니다.""" # LLM이 읽는 도구 설명
    try:
        # 안전한 계산을 위해 간단한 처리 (실제 서비스에선 더 정교한 로직 필요)
        return str(eval(expression))
    except Exception as e:
        return f"계산 오류: {str(e)}"

# 2. 도구 리스트 구성
tools = [complex_calculator]

In [ ]:
# 3. LLM 설정 및 도구 바인딩
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
# 도구 호출 기능이 최적화된 모델 버전 사용
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

# LLM에게 도구의 존재를 알려줍니다 (Bind)
llm_with_tools = llm.bind_tools(tools)

# 4. 실행 테스트 (도구 호출 판단 확인)
query = "12345 곱하기 67890의 결과는 뭐야?"
# 모델은 답변 대신 '도구 호출 요청'이 담긴 JSON 데이터를 생성합니다.
ai_msg = llm_with_tools.invoke(query)

print("--- [모델의 출력 결과 확인] ---")
print(f"Tool Calls: {ai_msg.tool_calls}") # 모델이 생성한 JSON 데이터



--- [모델의 출력 결과 확인] ---
Tool Calls: [{'name': 'complex_calculator', 'args': {'expression': '12345 * 67890'}, 'id': '995112ed-3608-4e9a-8d92-2b73645d0f0a', 'type': 'tool_call'}]


In [ ]:
from langchain_core.messages import ToolMessage

# 5. 실제 도구 실행
tool_call = ai_msg.tool_calls[0]
selected_tool = {"complex_calculator": complex_calculator}[tool_call["name"]]
tool_output = selected_tool.invoke(tool_call["args"])

print(f"[*] 도구 실행 결과: {tool_output}")

# 6. 도구 결과값을 모델에게 다시 전달하여 최종 답변 생성
final_response = llm_with_tools.invoke([
    ("human", query),
    ai_msg, # 모델의 도구 호출 요청 메시지
    ToolMessage(tool_output, tool_call_id=tool_call["id"]) # 도구 실행 결과 메시지
])

print(f"\n최종 답변: {final_response.content}")

[*] 도구 실행 결과: 838102050


ChatGoogleGenerativeAIError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 37.694231313s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-Fr

- 관찰 포인트  
    - **Description의 중요성**: @tool 아래에 적힌 독스트링(Docstring)이 모델에게 "이 도구는 언제 써야 한다"는 지시서 역할을 합니다.  
    - **JSON 추출**: LLM은 파이썬 코드를 직접 실행하는 것이 아니라, 실행에 필요한 **함수명과 인자(Arguments)**를 정확한 데이터 형식으로 뽑아내는 역할을 합니다.  
    - **데이터 순환**: 사용자 질문 → 모델의 도구 호출 요청 → 시스템의 도구 실행 → 모델의 최종 답변으로 이어지는 Loop 구조가 에이전트 행동의 기본입니다.  
    - 질문에 따라 모델이 ai_msg.content(텍스트)를 내보내는지, 아니면 ai_msg.tool_calls(JSON 데이터)를 내보내는지 비교해 보시는 것이 학습의 핵심입니다.  

2.2 복합 Custom Tool 정의 (날씨 API 시뮬레이션)  
2.2.1 날씨 도구 정의

- 단순한 텍스트 매칭을 넘어, 구조화된 인자를 받는 도구를 정의합니다.  

In [ ]:
# 1. 복합 파라미터를 가진 날씨 도구 정의
@tool
def get_current_weather(location: str, unit: str = "섭씨") -> str:
    """
    특정 지역의 실시간 날씨 정보를 가져옵니다.
    - location: 도시 이름 (예: 서울, 도쿄, 뉴욕 등)
    - unit: 온도 단위 (섭씨 또는 화씨)
    """ # 상세한 인자 설명은 LLM의 추출 정확도를 높입니다.

    # [실습용 시뮬레이션 데이터] 실제로는 OpenWeatherMap 등의 API를 호출합니다.
    weather_data = {
        "서울": {"temp": "25", "desc": "맑음"},
        "수원": {"temp": "23", "desc": "흐림"},
        "도쿄": {"temp": "28", "desc": "비"},
        "뉴욕": {"temp": "15", "desc": "쌀쌀함"}
    }

    result = weather_data.get(location, {"temp": "알 수 없음", "desc": "정보 없음"})
    return f"{location}의 현재 날씨는 {result['desc']}이며, 기온은 {result['temp']}{unit}입니다."

# 2. 도구 리스트 구성 (계산기 + 날씨)
tools = [get_current_weather]

2.2.2 Function Calling 실행 및 인자 추출 확인  

- LLM이 사용자의 문장에서 도구 실행에 필요한 정보를 어떻게 JSON으로 뽑아내는지 확인합니다.  

In [ ]:
# 3. LLM 설정 및 도구 바인딩
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash-latest', api_key=GOOGLE_API_KEY)
llm_with_tools = llm.bind_tools(tools)

# 4. 복합 질문 테스트
query = "수원의 현재 날씨를 화씨 단위로 알려줄래?"
ai_msg = llm_with_tools.invoke(query)

print("--- [모델의 인자 추출 결과] ---")
# 모델이 location='수원', unit='화씨'를 정확히 추출했는지 확인합니다.
print(ai_msg.tool_calls)

2.2.3. 도구 실행 및 결과 통합 (Manual Loop)  

- 모델이 요청한 도구를 실제로 실행하고 그 결과값을 다시 전달하여 최종 답변을 완성합니다.  

In [ ]:
# 5. 실행 루프 구현
if ai_msg.tool_calls:
    tool_call = ai_msg.tool_calls[0]

    # 도구 실행 (Observation)
    observation = get_current_weather.invoke(tool_call["args"])
    print(f"[*] 도구 호출 결과: {observation}")

    # 모델에게 결과 전달 및 최종 답변 생성
    final_response = llm_with_tools.invoke([
        ("human", query),
        ai_msg,
        ToolMessage(observation, tool_call_id=tool_call["id"])
    ])

    print(f"\n[최종 답변]: {final_response.content}")

- 관찰 포인트  

    - **멀티 파라미터 제어**: 모델이 location과 unit이라는 두 개의 정보를 문장에서 동시에 찾아내어 적절한 도구 인자로 매핑하는 기능  
    - **도구 간 우선순위**: 여러 도구(계산기, 날씨 등)가 있을 때 질문의 의도에 맞춰 어떤 도구를 선택할지 결정하는 모델의 판단력을 관찰  
    - **실시간 데이터 통합**: 정적인 학습 데이터가 아닌, 도구 실행 결과라는 외부 컨텍스트를 답변에 녹여내는 과정  